### Supplement: 2-Structured Outputs with Pydantic Deep Dive

This supplement notebook breaks down `2-structured.py` cell by cell. It covers **Structured Outputs** using Pydantic and the OpenAI Python SDK:

1. **What is Pydantic and `BaseModel`?**: Defining data schemas with strict types.
2. **Schema Generation**: How `CalendarEvent` is converted to a JSON Schema under the hood.
3. **`create` vs. `parse`**: Understanding `client.chat.completions.parse(...)`.
4. **`message.content` vs. `message.parsed`**:
   - `message.content`: The raw JSON string returned by the LLM.
   - `message.parsed`: The deserialized, validated Pydantic object!
5. **Accessing Typed Attributes & Converting to Python Dicts**: Dot-notation vs. `.model_dump()`.
6. **Cheat Sheet**: `model_json_schema()` vs. `model_dump()` vs. `json.loads()` vs. `.parsed`.

---

##### Architectural Flow:
```
1. Define Pydantic Schema:
   class CalendarEvent(BaseModel):
       name: str
       date: str
       participants: list[str]
            │
            ▼
2. SDK calls .model_json_schema(), tightens it for strict mode,
   and puts it in the request body sent to OpenAI
            │
            ▼
3. OpenAI API uses Constrained Sampling (the emitted JSON is
   guaranteed to match the schema — unless the model refuses,
   or generation is cut off by a token limit)
            │
            ▼
4. Response arrives back at SDK:
   ├── message.content -> raw JSON string: '{"name": "...", "date": "...", ...}'
   └── message.parsed  -> CalendarEvent(name='...', date='...', participants=[...])
```

> **Note on `.beta.`**: older tutorials call `client.beta.chat.completions.parse(...)`. Structured-output parsing has since graduated out of beta, so this notebook uses `client.chat.completions.parse(...)` — matching `2-structured.py`. The `.beta.` path still exists in the installed SDK (openai v2.1.0) and is functionally equivalent, so older code keeps working. The one visible difference is cosmetic: `.beta.` prints its type as `ParsedChatCompletion[CalendarEvent]`, while the stable path prints `ParsedChatCompletion[TypeVar]` (see Section 3). Prefer the non-`beta` path in anything new.

#### 1. Imports and Environment Setup

Notice we import:
- `OpenAI`: The API client class.
- `BaseModel`, `Field`: From `pydantic`. `BaseModel` is the base class for defining data contracts and schemas.

In [2]:
import os
import json
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# Load API key
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    load_dotenv(r"C:\Users\ashut\ML_Practice\LLM_Learning_Sandbox\.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Client initialized successfully.")
print("Pydantic BaseModel class:", BaseModel)


Client initialized successfully.
Pydantic BaseModel class: <class 'pydantic.main.BaseModel'>


#### 2. Defining the Schema Class: `CalendarEvent(BaseModel)`

##### What is `BaseModel`?
`BaseModel` is Pydantic's core class. When you subclass `BaseModel`:
- It parses and validates data according to type hints (`str`, `list[str]`, etc.).
- It can automatically export a standard JSON Schema via `.model_json_schema()`.
- It allows serialization back to dicts via `.model_dump()`.

Below we call `.model_json_schema()` **purely for inspection**, so you can see the shape the SDK will derive from your class. Note that nothing in this notebook sends that `schema` variable anywhere — the SDK regenerates it internally when you pass `response_format=CalendarEvent` in Section 3. The next cell explains exactly who consumes it.

In [3]:
class CalendarEvent(BaseModel):
    name: str = Field(description="The name or title of the event")
    date: str = Field(description="The date or day of the event")
    participants: list[str] = Field(description="List of attendee names")

print("Class name:", CalendarEvent.__name__)
print("Inherits from:", [b.__name__ for b in CalendarEvent.__bases__])
print("Declared fields:", list(CalendarEvent.model_fields.keys()))

print("\n--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---")
schema = CalendarEvent.model_json_schema()
print(json.dumps(schema, indent=2))


Class name: CalendarEvent
Inherits from: ['BaseModel']
Declared fields: ['name', 'date', 'participants']

--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---
{
  "properties": {
    "name": {
      "description": "The name or title of the event",
      "title": "Name",
      "type": "string"
    },
    "date": {
      "description": "The date or day of the event",
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "description": "List of attendee names",
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}


##### Wait — who actually *uses* this JSON Schema, and for what?

This is the single most confusing part of structured outputs, so let's be precise.

**You never call `.model_json_schema()` yourself in normal usage.** The cell above is a teaching X-ray. The schema's real consumer is **OpenAI's inference server**, not your Python code.

Here is the actual chain of custody:

1. You pass the **class itself** (not a schema) to the SDK: `response_format=CalendarEvent`.
2. **The SDK** — inside `openai/lib/_parsing/_completions.py` — calls `CalendarEvent.model_json_schema()` for you.
3. **The SDK tightens it for strict mode**: it recursively injects `"additionalProperties": false` into every object and wraps the result with a `name` and `strict: true`.
4. That payload goes over the wire in the request body.
5. **OpenAI's server** compiles the schema into a grammar and uses it for **constrained decoding** — at each step, tokens that would break the schema are masked out of the sampling distribution. The model is *mechanically unable* to emit a wrong field name, a wrong type, or a missing required field.

So the answer to *"where is this JSON used, by whom, for what?"* is: **by OpenAI's token sampler, to make invalid output impossible.** It is a contract shipped to the model, not data for your program.

##### Important: the raw Pydantic schema is *not* byte-for-byte what gets sent

The printout above is Pydantic's generic export. Compare it to what the SDK actually transmits:

```python
from openai.lib._parsing._completions import type_to_response_format_param
print(json.dumps(type_to_response_format_param(CalendarEvent), indent=2))
```

```jsonc
{
  "type": "json_schema",
  "json_schema": {
    "schema": {
      "properties": { /* ...same as above... */ },
      "required": ["name", "date", "participants"],
      "title": "CalendarEvent",
      "type": "object",
      "additionalProperties": false   // <-- ADDED by the SDK for strict mode
    },
    "name": "CalendarEvent",          // <-- ADDED
    "strict": true                    // <-- ADDED
  }
}
```

Two practical consequences:
- The `description=` text you wrote in each `Field(...)` **does** survive into the schema, so it reaches the model and acts as a per-field prompt. Descriptive `Field` descriptions genuinely improve extraction quality.
- Strict mode forbids optional/extra keys. Every field is required, which is why you model "maybe missing" as `Optional[str]` (i.e. `str | None`) rather than by omitting the field.

#### 3. The API Call: `client.chat.completions.parse(...)`

##### Why `parse(...)` instead of `create(...)`?
- **`create(...)`**: You get back a standard `ChatCompletion`. `message.content` is just a `str`, and `message.parsed` does not exist. To get an object you must deserialize it yourself:
  ```python
  data = json.loads(completion.choices[0].message.content)  # -> plain dict
  event = CalendarEvent(**data)                             # -> validate by hand
  ```
- **`parse(...)`**: A high-level helper in the OpenAI SDK that:
  1. Converts your Pydantic class to a JSON Schema and sends it with `strict: true`.
  2. The model generates strictly conforming JSON tokens via constrained decoding.
  3. The SDK automatically validates the JSON and instantiates your `CalendarEvent` class!
  4. The instantiated object is placed in `completion.choices[0].message.parsed`.

> **A precise distinction.** `create()` is not inherently "unsafe JSON" — it also accepts `response_format={"type": "json_schema", ...}` with strict mode, giving the same generation guarantee. The catch is that you must hand-write that schema dict and do your own `json.loads()` + validation. So `parse()` is not buying you *reliability the API otherwise lacks*; it is buying you **the Pydantic-class-to-schema conversion on the way out, and the deserialization on the way back.**
>
> The older "JSON mode" (`response_format={"type": "json_object"}`) is the genuinely weaker option: it guarantees only *syntactically valid* JSON, with no control over which fields appear. That is the case where "hope the model didn't invent fields" actually applies.

##### Don't be thrown by `ParsedChatCompletion[TypeVar]` in the output below

The printed type reads `ParsedChatCompletion[TypeVar]` rather than the more informative `ParsedChatCompletion[CalendarEvent]` you'd see from the older `.beta.` path. This is **purely cosmetic** — the non-beta `parse()` doesn't bind the generic parameter into the runtime repr.

Nothing about the behaviour changes, as Section 4 confirms: `message.parsed` is a genuine `CalendarEvent` and `isinstance(message.parsed, CalendarEvent)` is `True`. Static type checkers still infer the correct type in your editor; only this one `print(type(...))` string is less specific.

In [4]:
prompt_messages = [
    {"role": "system", "content": "Extract the event information."},
    {
        "role": "user",
        "content": "Alice and Bob are going to a science fair on Friday.",
    },
]

completion = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=prompt_messages,
    response_format=CalendarEvent,
)

print("Parsed completion call successful!")
print("Completion object type:", type(completion))
print("Choices length:", len(completion.choices))

Parsed completion call successful!
Completion object type: <class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
Choices length: 1


#### 4. Comparing `message.content` vs. `message.parsed`

This is the most critical distinction in Structured Outputs:
1. **`message.content`**: Contains the **raw JSON string** sent across the wire by the LLM.
2. **`message.parsed`**: Contains the **instantiated Python Pydantic object** (`CalendarEvent`).

Both are populated on a successful call — `parsed` is simply the SDK having already done `json.loads()` + Pydantic validation on `content` for you.

The cell also prints **`message.refusal`**, which is the escape hatch in the schema guarantee. If the model declines the request on safety grounds, it returns a plain-text refusal *instead of* schema-conforming JSON: `refusal` holds that text, and `parsed` is `None`. In production this is what you branch on:

```python
if message.refusal:
    handle_refusal(message.refusal)
else:
    event = message.parsed
```

Let's inspect all three with `type()` and `repr()`:

In [5]:
message = completion.choices[0].message

print("=== 1. message.content (Raw Wire JSON) ===")
print("Type:", type(message.content))
print("Raw string value:", repr(message.content))

print("\n=== 2. message.parsed (Deserialized Pydantic Object) ===")
print("Type:", type(message.parsed))
print("Is instance of CalendarEvent?:", isinstance(message.parsed, CalendarEvent))
print("Object representation:", repr(message.parsed))

print("\n=== 3. message.refusal ===")
print("Refusal status:", message.refusal)


=== 1. message.content (Raw Wire JSON) ===
Type: <class 'str'>
Raw string value: '{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

=== 2. message.parsed (Deserialized Pydantic Object) ===
Type: <class '__main__.CalendarEvent'>
Is instance of CalendarEvent?: True
Object representation: CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

=== 3. message.refusal ===
Refusal status: None


##### Visualizing the Nested Hierarchy: where `message.parsed` sits

This is the same walk as the `.content` tree in `Supplement_1-basic.ipynb` (Section 5), redrawn for a `parse()` response. `.parsed` sits next to `.content`, and it has children of its own: the fields you declared in `CalendarEvent`.

```
completion                                             <class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
  │
  ├── .choices                                         <class 'list'>
  │     │
  │     └── [0]                                        <class 'openai.types.chat.parsed_chat_completion.ParsedChoice[TypeVar]'>
  │           │
  │           ├── .index: 0                            <class 'int'>
  │           ├── .finish_reason: 'stop'               <class 'str'>
  │           └── .message                             <class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletionMessage[TypeVar]'>
  │                 │
  │                 ├── .role: 'assistant'             <class 'str'>
  │                 ├── .content: '{"name":...}'       <class 'str'> (the raw JSON text the model generated)
  │                 ├── .refusal: None                 <class 'NoneType'>
  │                 └── .parsed                        <class '__main__.CalendarEvent'> (YOUR class, built from .content by the SDK)
  │                       │
  │                       ├── .name: 'Science Fair'    <class 'str'>
  │                       ├── .date: 'Friday'          <class 'str'>
  │                       └── .participants            <class 'list'>
  │                             │
  │                             ├── [0]: 'Alice'       <class 'str'>
  │                             └── [1]: 'Bob'         <class 'str'>
```

What's different from the `create()` tree:
- **Each class on the path gets a `Parsed` prefix.** `ChatCompletion` becomes `ParsedChatCompletion`, `Choice` becomes `ParsedChoice`, and `ChatCompletionMessage` becomes `ParsedChatCompletionMessage`. Each one is a subclass of the original. `ParsedChatCompletionMessage` is the class that adds the `parsed` field. The other two change their child's type so that the path leads down to it.
- **Everything above `.parsed` uses OpenAI's classes. `.parsed` itself is your `CalendarEvent`.** Its children are exactly the three fields you declared.
- **The two parts are built differently.** The SDK builds the outer layers with the lenient `construct_type`, which checks no types (see `Supplement_1` Section 4b). It builds `.parsed` with `CalendarEvent.model_validate_json(message.content)`, which runs full Pydantic validation.

#### 5. Accessing Typed Attributes & Converting to Python Dict

Because `event` is a `CalendarEvent` object:
- You get typed attribute access (dot-notation) with IDE autocompletion: `event.name`, `event.date`, `event.participants`.
- `event.participants` is a genuine Python `list` of strings!
- You can convert the object to a standard Python dictionary using `event.model_dump()`.

In [6]:
event: CalendarEvent = message.parsed

print("--- Accessing Typed Attributes ---")
print(f"event.name:         {event.name} (type: {type(event.name)})")
print(f"event.date:         {event.date} (type: {type(event.date)})")
print(f"event.participants: {event.participants} (type: {type(event.participants)})")
print(f"First participant:  {event.participants[0]} (type: {type(event.participants[0])})")

print("\n--- Converting to Standard Python Dictionary (.model_dump()) ---")
event_dict = event.model_dump()
print("Type of event_dict:", type(event_dict))
print("Dictionary content:", event_dict)


--- Accessing Typed Attributes ---
event.name:         Science Fair (type: <class 'str'>)
event.date:         Friday (type: <class 'str'>)
event.participants: ['Alice', 'Bob'] (type: <class 'list'>)
First participant:  Alice (type: <class 'str'>)

--- Converting to Standard Python Dictionary (.model_dump()) ---
Type of event_dict: <class 'dict'>
Dictionary content: {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


#### 6. Visualizing the Complete Structured Response Object

Let's dump the entire `completion` object to inspect everything OpenAI returned, including token usage and choice metadata:

In [7]:
full_dict = completion.model_dump()

print("Full Completion Dictionary:")
print(json.dumps(full_dict, indent=2))


Full Completion Dictionary:
{
  "id": "chatcmpl-EPCKDPLr9Le2JXNU0xgieEeXtUYED",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Science Fair\",\"date\":\"Friday\",\"participants\":[\"Alice\",\"Bob\"]}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Science Fair",
          "date": "Friday",
          "participants": [
            "Alice",
            "Bob"
          ]
        }
      }
    }
  ],
  "created": 1789674285,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 414,
    "prompt_tokens": 117,
    "total_tokens": 531,
    "completion_tokens_details": {
      "accepted_predictio

c:\Users\ashut\anaconda3\envs\General_env\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CalendarEvent(name='Scien...ipants=['Alice', 'Bob']), input_type=CalendarEvent])
  return self.__pydantic_serializer__.to_python(


---

#### 7. Cheat Sheet: `model_json_schema()` vs. `model_dump()` vs. `json.loads()` vs. `.parsed`

These names blur together because they all involve "JSON" — but they move in **different directions** and belong to **different libraries**. Sort them by *what goes in* and *what comes out*.

##### The one distinction that unlocks the rest

- `model_json_schema()` describes the **shape** — it operates on the **class** and produces a *description of a type*. It contains no data. It travels **outward to the model**.
- `model_dump()` / `model_dump_json()` carry the **data** — they operate on an **instance** and produce *values*. They travel **outward to your code, a file, or another service**.

> A schema is the mould; a dump is the casting. `CalendarEvent.model_json_schema()` works without any event ever existing, whereas `event.model_dump()` needs a real `event`.

##### The same idea, in plain words

Think of `CalendarEvent` as a **blank paper form** with three boxes to fill in: *Name*, *Date* and *Participants*.

- The **class** (`CalendarEvent`) is the blank form. An **instance** (`event`) is one copy of that form with the boxes filled in.
- **`model_json_schema()` describes the blank form.** It says things like "the *name* box takes text" and "the *participants* box takes a list of text". It can't mention "Science Fair", because a blank form has no answers on it. That's what "it contains no data" means.
- **`model_dump()` reads the answers off one filled-in form.** It says "name is Science Fair". There has to be a filled-in form to read from, which is why it works on an *instance* and not on the class.

##### Seeing it in code

```python
# 1. Ask the CLASS for its shape. No event exists yet, and that's fine.
schema = CalendarEvent.model_json_schema()
print(schema["properties"]["name"])
# -> {'description': 'The name or title of the event', 'title': 'Name', 'type': 'string'}
#    It says "name is a string". It does NOT say "Science Fair". Only rules, no data.

# 2. Fill in one form: create a real event.
event = CalendarEvent(name="Science Fair", date="Friday", participants=["Alice", "Bob"])

# 3. Ask the INSTANCE for its data.
print(event.model_dump())
# -> {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}
#    Now there are real values, and no rules: no 'string', no 'array'.

# 4. A second event has DIFFERENT data but the SAME shape.
other = CalendarEvent(name="Book Club", date="Monday", participants=["Carol"])
print(other.model_dump())
# -> {'name': 'Book Club', 'date': 'Monday', 'participants': ['Carol']}
print(other.model_json_schema() == event.model_json_schema())
# -> True   (the shape belongs to the class, so every event shares it)

# 5. Try to dump the class itself. There is no event, so there is nothing to dump.
CalendarEvent.model_dump()
# -> TypeError: BaseModel.model_dump() missing 1 required positional argument: 'self'
```

##### Where each one travels

- **The schema goes to the model *before* it writes anything.** You never send it yourself. When you call `parse(response_format=CalendarEvent)`, the SDK creates the schema and puts it in the request. OpenAI's server then uses it as the rules for which tokens the model may write. In the form analogy, this is handing someone the blank form and saying "fill in exactly this".
- **The dump goes wherever *you* send the data, *after* you have it.** This part is your own code:

```python
data = event.model_dump()

# 1. Your own code uses the values.
print("Invite:", ", ".join(data["participants"]))
# -> Invite: Alice, Bob

# 2. A file.
with open("event.json", "w") as f:
    f.write(event.model_dump_json())

# 3. Another service (not run here).
# requests.post("https://example.com/events", json=data)
```

##### Full reference, one call at a time

Every example below uses the same event as Section 5:

```python
event = CalendarEvent(name="Science Fair", date="Friday", participants=["Alice", "Bob"])
```

##### 1. `CalendarEvent.model_json_schema()`

- **Input → Output:** **class** → `dict` describing the *shape*
- **Library:** Pydantic
- **Where it appears here:** Section 2 — sent by the SDK to OpenAI to constrain decoding

**Example**

```python
# Called on the CLASS. No event is needed.
schema = CalendarEvent.model_json_schema()

print(type(schema))
# -> <class 'dict'>

print(schema["required"])
# -> ['name', 'date', 'participants']

print(schema["properties"]["participants"])
# -> {'description': 'List of attendee names', 'items': {'type': 'string'}, 'title': 'Participants', 'type': 'array'}
```

Read the last line as a rule: "*participants* must be an array, and every item must be a string". It lists fields and types, but no names like "Alice". The `description` you wrote in `Field(...)` is in there too, which is how it reaches the model.

##### 2. `event.model_dump()`

- **Input → Output:** **instance** → Python `dict` (values stay Python objects)
- **Library:** Pydantic
- **Where it appears here:** Section 5 — `{'name': 'Science Fair', ...}`

**Example**

```python
d = event.model_dump()

print(d)
# -> {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}

print(type(d))
# -> <class 'dict'>

# "Values stay Python objects": participants is a real Python list, not text.
print(type(d["participants"]))
# -> <class 'list'>

# So you can change it like any other list.
d["participants"].append("Carol")
print(d["participants"])
# -> ['Alice', 'Bob', 'Carol']

# The dict is a copy. Changing it does not change the event.
print(event.participants)
# -> ['Alice', 'Bob']
```

The single quotes in the output are a clue that this is a Python `dict`, not JSON text. JSON always uses double quotes.

##### 3. `event.model_dump_json()`

- **Input → Output:** **instance** → JSON **`str`** (skips the dict step)
- **Library:** Pydantic
- **Where it appears here:** not used here; ≈ `json.dumps(event.model_dump())`

**Example**

```python
s = event.model_dump_json()

print(s)
# -> {"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}

print(type(s))
# -> <class 'str'>

# It's text now, so indexing gives you one character...
print(s[0])
# -> {

# ...and you can't look up keys any more.
s["name"]
# -> TypeError: string indices must be integers, not 'str'

# The ≈ version gives the same JSON, with a space after each : and ,
print(json.dumps(event.model_dump()))
# -> {"name": "Science Fair", "date": "Friday", "participants": ["Alice", "Bob"]}
```

Use this when the data is leaving Python, for example to write a `.json` file or to go in an HTTP request body. "Skips the dict step" means Pydantic writes the text directly, instead of building a dict first and converting that.

##### 4. `json.loads(s)`

- **Input → Output:** JSON **`str`** → `dict` / `list`
- **Library:** stdlib `json`
- **Where it appears here:** the manual path you'd need after `create()`

**Example**

```python
# The kind of text message.content holds after create().
raw = '{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

data = json.loads(raw)
print(type(data))
# -> <class 'dict'>

# A dict gives you square brackets...
print(data["name"])
# -> Science Fair

# ...but not dots. Only a CalendarEvent has .name.
data.name
# -> AttributeError: 'dict' object has no attribute 'name'

# The second manual step: turn the dict into your class.
event = CalendarEvent(**data)
print(event.name)
# -> Science Fair

# If the JSON text is a list, you get a list back.
print(json.loads('["Alice", "Bob"]'))
# -> ['Alice', 'Bob']
```

`json.loads` only reads the text. It has never heard of `CalendarEvent`, so it checks nothing. For example, `json.loads('{"name": "Science Fair"}')` works fine even though `date` and `participants` are missing. The problem only shows up at the next step, when `CalendarEvent(**data)` raises `ValidationError: 2 validation errors for CalendarEvent ... Field required`.

##### 5. `json.dumps(obj)`

- **Input → Output:** `dict` → JSON **`str`**
- **Library:** stdlib `json`
- **Where it appears here:** Sections 2 and 6 — pretty-printing only

**Example**

```python
d = {"name": "Science Fair", "participants": ["Alice", "Bob"]}

# Printing the dict directly shows Python style: single quotes.
print(d)
# -> {'name': 'Science Fair', 'participants': ['Alice', 'Bob']}

# json.dumps turns it into JSON text: double quotes, and the result is a str.
print(json.dumps(d))
# -> {"name": "Science Fair", "participants": ["Alice", "Bob"]}

# indent=2 is the pretty-printing used in Sections 2 and 6.
print(json.dumps(d, indent=2))
# -> {
#      "name": "Science Fair",
#      "participants": [
#        "Alice",
#        "Bob"
#      ]
#    }

# It only understands plain Python types, not your class.
json.dumps(event)
# -> TypeError: Object of type CalendarEvent is not JSON serializable
```

That last error is why you call `model_dump()` first: it turns `event` into a plain dict that `json.dumps` can handle. Or you can use `event.model_dump_json()`, which does both steps at once.

##### 6. `message.content`

- **Input → Output:** *(not a call)* the raw JSON **`str`** the model emitted
- **Library:** OpenAI SDK field
- **Where it appears here:** Section 4

**Example**

```python
message = completion.choices[0].message

print(repr(message.content))
# -> '{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

print(type(message.content))
# -> <class 'str'>

# It's text, so the first 8 items are just characters.
print(message.content[:8])
# -> {"name":
```

There are no brackets `()` because nothing is being called. It's a value the SDK stored: the exact text the model wrote. With `create()` this string is all you get, so you would carry on with example 4.

##### 7. `message.parsed`

- **Input → Output:** *(not a call)* the ready-made `CalendarEvent` **instance**
- **Library:** OpenAI SDK, via `parse()`
- **Where it appears here:** Sections 4 and 5 — the payoff

**Example**

```python
event = message.parsed

print(repr(event))
# -> CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

# Dot access works straight away.
print(event.name)
# -> Science Fair
print(event.participants[1])
# -> Bob

# It is exactly what you'd get by converting message.content yourself.
print(CalendarEvent.model_validate_json(message.content) == message.parsed)
# -> True
```

The last line shows what `parse()` did for you. `model_validate_json` reads the JSON text and builds a checked `CalendarEvent` in one step. Those are the two manual steps from example 4 (`json.loads`, then `CalendarEvent(**data)`), already done.

##### 8. `response.json()`

- **Input → Output:** HTTP response body → `dict`
- **Library:** **`requests` / `httpx`** — *a different library entirely*
- **Where it appears here:** never in your code; runs **inside** the SDK

**Example** (runs offline, because we build a fake HTTP response by hand)

```python
import httpx2

# What arrives from the network is raw bytes, not a dict.
response = httpx2.Response(200, content=b'{"id": "chatcmpl-123", "object": "chat.completion"}')

print(type(response.content))
# -> <class 'bytes'>

data = response.json()
print(data)
# -> {'id': 'chatcmpl-123', 'object': 'chat.completion'}
print(type(data))
# -> <class 'dict'>

# It gives the same result as json.loads on the body text.
print(data == json.loads(response.text))
# -> True
```

This example uses `httpx2`, the successor to `httpx` from the same author. It's the HTTP library used by the OpenAI SDK installed here (v3.14.1). `requests` and `httpx` have the same `.json()` method. Inside the SDK, `response.json()` turns the raw body into a dict like this one. The SDK then builds the `ParsedChatCompletion` object from that dict.

##### On `response.json()`

If you have seen this elsewhere and mentally filed it with the others, separate it now. It belongs to HTTP clients like `requests` and `httpx`, and means *"parse this HTTP response body as JSON"*. It is roughly `json.loads(response.text)` plus encoding handling. It has nothing to do with Pydantic.

You never write it yourself when using the OpenAI SDK — but it **does** run, one layer down: the SDK calls httpx's `response.json()` (`httpx2` in the installed version) internally to turn the raw HTTP body into a dict, before building the typed response object from that dict. `Supplement_1-basic.ipynb`, Section 4 traces that exact step. So the rule is *"not in your code"*, not *"never happens"*.

##### Why Section 6 also calls `.model_dump()`

A neat reinforcement: in Section 6 we call `completion.model_dump()` on the **response object**, not on our own event. That works because the OpenAI SDK's own response classes (`ChatCompletion`, `ParsedChatCompletion`, …) are *themselves* Pydantic `BaseModel` subclasses. Same method, same direction — instance to dict — just applied to a class the SDK authored instead of one you wrote.

##### The one-sentence version

> `model_json_schema()` sends a **shape** outward so the model cannot produce malformed output; `model_dump()` / `model_dump_json()` take a **populated object** and convert it to a dict or string for your own use; `json.loads()` is the generic, non-Pydantic way to turn a JSON string into a dict — which is exactly the step `parse()` performs on your behalf.